In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
consumption = pd.read_parquet(
    "../data/raw/consumption.parquet"
)

consumption.head()

,date,territory_id,category,value
0,2023-01,1,Продовольствие,7692
1,2023-01,1,Здоровье,1271
2,2023-01,1,Маркетплейсы,2505
3,2023-01,1,Общественное питание,1142
5,2023-01,1,Транспорт,1718


In [3]:
spending_wide = (
    consumption[
        consumption["category"] != "Все категории"
    ]
    .pivot(
        index=["territory_id", "date"],
        columns="category",
        values="value"
    )
    .reset_index()
)

In [4]:
spending_wide.head()

category,territory_id,date,Здоровье,Маркетплейсы,Общественное питание,Продовольствие,Транспорт
0,1,2023-01,1271,2505,1142,7692,1718
1,1,2023-02,1276,2748,1095,8259,1704
2,1,2023-03,1410,3052,1256,8873,1847
3,1,2023-04,1277,2850,1171,8648,1784
4,1,2023-05,1295,2994,1315,8738,1817


In [5]:
spending_wide = spending_wide.rename(
    columns={
        "Продовольствие": "food",
        "Здоровье": "health",
        "Маркетплейсы": "marketplaces",
        "Общественное питание": "catering",
        "Транспорт": "transport"
    }
)

In [6]:
spending_wide.head()

category,territory_id,date,health,marketplaces,catering,food,transport
0,1,2023-01,1271,2505,1142,7692,1718
1,1,2023-02,1276,2748,1095,8259,1704
2,1,2023-03,1410,3052,1256,8873,1847
3,1,2023-04,1277,2850,1171,8648,1784
4,1,2023-05,1295,2994,1315,8738,1817


In [7]:
spending_columns = [
    "food",
    "health",
    "marketplaces",
    "catering",
    "transport"
]

spending_wide["total_spending"] = (
    spending_wide[spending_columns].sum(axis=1)
)

In [8]:
for col in spending_columns:
    spending_wide[f"{col}_share"] = (
        spending_wide[col]
        / spending_wide["total_spending"]
    )

In [9]:
share_columns = [
    "food_share",
    "health_share",
    "marketplaces_share",
    "catering_share",
    "transport_share"
]

spending_wide[share_columns].sum(axis=1).describe()

count    5.052100e+04
mean     1.000000e+00
std      6.521944e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64

In [10]:
spending_wide[
    spending_wide["territory_id"] == 1
].head(10)

category,territory_id,date,health,marketplaces,catering,food,transport,total_spending,food_share,health_share,marketplaces_share,catering_share,transport_share
0,1,2023-01,1271,2505,1142,7692,1718,14328,0.536851,0.088707,0.174832,0.079704,0.119905
1,1,2023-02,1276,2748,1095,8259,1704,15082,0.547606,0.084604,0.182204,0.072603,0.112982
2,1,2023-03,1410,3052,1256,8873,1847,16438,0.539786,0.085777,0.185667,0.076408,0.112362
3,1,2023-04,1277,2850,1171,8648,1784,15730,0.549777,0.081182,0.181182,0.074444,0.113414
4,1,2023-05,1295,2994,1315,8738,1817,16159,0.540751,0.080141,0.185284,0.081379,0.112445
5,1,2023-06,1244,3175,1380,8911,1918,16628,0.535903,0.074814,0.190943,0.082993,0.115348
6,1,2023-07,1336,3463,1510,9372,2060,17741,0.528268,0.075306,0.195198,0.085114,0.116115
7,1,2023-08,1400,3632,1507,9435,2112,18086,0.521674,0.077408,0.200818,0.083324,0.116775
8,1,2023-09,1363,3847,1444,8913,1949,17516,0.508849,0.077815,0.219628,0.082439,0.111270
9,1,2023-10,1409,4172,1398,9108,1917,18004,0.505888,0.078260,0.231726,0.077649,0.106476


In [11]:
market_access = pd.read_parquet(
    "../data/raw/market_access.parquet"
)

market_access.head()

,territory_id,market_access
0,1,309.6
1,2,322.8
2,3,315.2
3,4,315.5
4,5,320.2


In [12]:
market_access.info()

<class 'pandas.DataFrame'>
RangeIndex: 2571 entries, 0 to 2570
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   territory_id   2571 non-null   int32  
 1   market_access  2571 non-null   float64
dtypes: float64(1), int32(1)
memory usage: 30.3 KB


In [13]:
market_access["territory_id"].nunique()

2571

In [14]:
features = spending_wide.merge(
    market_access,
    on="territory_id",
    how="left"
)

In [15]:
features.head()

,territory_id,date,health,marketplaces,catering,food,transport,total_spending,food_share,health_share,marketplaces_share,catering_share,transport_share,market_access
0,1,2023-01,1271,2505,1142,7692,1718,14328,0.536851,0.088707,0.174832,0.079704,0.119905,309.6
1,1,2023-02,1276,2748,1095,8259,1704,15082,0.547606,0.084604,0.182204,0.072603,0.112982,309.6
2,1,2023-03,1410,3052,1256,8873,1847,16438,0.539786,0.085777,0.185667,0.076408,0.112362,309.6
3,1,2023-04,1277,2850,1171,8648,1784,15730,0.549777,0.081182,0.181182,0.074444,0.113414,309.6
4,1,2023-05,1295,2994,1315,8738,1817,16159,0.540751,0.080141,0.185284,0.081379,0.112445,309.6


In [16]:
features.shape

(50521, 14)

In [17]:
features.isna().sum()

territory_id            0
date                    0
health                  0
marketplaces            0
catering                0
food                    0
transport               0
total_spending          0
food_share              0
health_share            0
marketplaces_share      0
catering_share          0
transport_share         0
market_access         398
dtype: int64

In [18]:
features["territory_id"].nunique()

2190

In [19]:
features["date"].nunique()

24

In [20]:
features.groupby("date")["territory_id"].nunique().describe()

count      24.000000
mean     2105.041667
std        21.416124
min      2072.000000
25%      2088.500000
50%      2101.500000
75%      2119.500000
max      2149.000000
Name: territory_id, dtype: float64

In [21]:
features.to_parquet(
    "../data/processed/features_basic.parquet",
    index=False
)

In [22]:
print("Размер таблицы:", features.shape)

print("\nКоличество муниципалитетов:")
print(features["territory_id"].nunique())

print("\nКоличество месяцев:")
print(features["date"].nunique())

print("\nПериод:")
print(features["date"].min(), "—", features["date"].max())

print("\nПропуски:")
print(features.isna().sum())

print("\nКоличество муниципалитетов по месяцам:")
print(features.groupby("date")["territory_id"].nunique().describe())

Размер таблицы: (50521, 14)

Количество муниципалитетов:
2190

Количество месяцев:
24

Период:
2023-01 — 2024-12

Пропуски:
territory_id            0
date                    0
health                  0
marketplaces            0
catering                0
food                    0
transport               0
total_spending          0
food_share              0
health_share            0
marketplaces_share      0
catering_share          0
transport_share         0
market_access         398
dtype: int64

Количество муниципалитетов по месяцам:
count      24.000000
mean     2105.041667
std        21.416124
min      2072.000000
25%      2088.500000
50%      2101.500000
75%      2119.500000
max      2149.000000
Name: territory_id, dtype: float64


In [23]:
missing_market = (
    features[features["market_access"].isna()]
    .groupby("territory_id")
    .size()
)

print("Количество муниципалитетов без market_access:")
print(missing_market.shape[0])

print("\nКоличество наблюдений без market_access:")
print(missing_market.sum())

print("\nСколько месяцев отсутствует market_access:")
print(missing_market.value_counts().sort_index())

Количество муниципалитетов без market_access:
20

Количество наблюдений без market_access:
398

Сколько месяцев отсутствует market_access:
5      1
12     4
15     1
19     1
23     1
24    12
Name: count, dtype: int64
